In [1]:
import dataclasses

import numpy as np
import pandas as pd


from gbp.loaders.contracts import GraphLoaderConfig
from gbp.loaders.dataloader_graph import DataLoaderGraph
from gbp.loaders.dataloader_mock import DataLoaderMock
from gbp.build.pipeline import build_model

mock_config = {
    "n_stations": 10,
    "n_depots": 2,
    "n_timestamps": 72,
    "time_freq": "h",
    "start_date": "2025-01-01",
    "ebike_fraction": 0.3,
    "depot_capacity": 200,
    "seed": 42,
}
mock_source = DataLoaderMock(mock_config, n_trucks=3, truck_capacity_bikes=20)
graph_loader = DataLoaderGraph(mock_source, GraphLoaderConfig())
raw_model = graph_loader.load()
resolved_with_supply = build_model(raw_model)
resolved_model = dataclasses.replace(resolved_with_supply, supply=None)

2026-05-13 11:42:50 [info     ] load_start                     loader=graph_core
2026-05-13 11:42:51 [debug    ] source_validated               loader=graph_core
2026-05-13 11:42:51 [debug    ] observed_flow_built            loader=graph_core rows=305
2026-05-13 11:42:51 [debug    ] observed_inventory_built       loader=graph_core rows=60
2026-05-13 11:42:51 [info     ] load_done                      facilities=12 loader=graph_core


In [ ]:
mock_source = DataLoaderMock(mock_config, n_trucks=3, truck_capacity_bikes=20)

### Initially we should have only df_trips
### Some of the tables we can generate using df_trips
### Some of the tables we can generate using mock_functions
### Thus - we have two types of tables

# self.df_stations
# self.df_station_capacities
# self.df_station_costs, 

# self.df_depots
# self.df_depot_capacities
# self.df_depot_costs, 

# self.timestamps

# self.inventory_initial,
# self.df_telemetry_ts, 
# self.df_trips

# self.df_resources
# self.df_resource_capacities
# self.df_truck_rates

In [ ]:
### Выпиши все таблицы!!!

# temporal = self._build_temporal()
# entities = self._build_entities()
# behavior = self._build_behavior(entities)
# distance_data = self._build_distance_matrix(entities) if self._config.build_edges else {}
# resources = self._build_resources(entities)

# observations: dict[str, pd.DataFrame | None] = {}
# if self._config.build_observations:
#     observations = self._build_observations(entities)

# registry = AttributeRegistry()
# node_params = self._build_node_parameters(registry)
# self._register_facility_costs(registry, temporal)
# self._register_resource_costs(registry, entities)

# all_tables = {
#     **temporal,
#     **entities.tables,
#     **behavior,
#     **distance_data,
#     **node_params,
#     **resources,
#     **{k: v for k, v in observations.items() if v is not None},
# }
# return RawModelData(
#     **{k: v for k, v in all_tables.items() if v is not None},
#     attributes=registry,
# )

In [ ]:
def load(self) -> RawModelData:
    """Load source tables and assemble ``RawModelData``.

    Validation and derivation of optional tables happen later, inside
    ``build_model``.  The returned raw model is cached on the loader and
    also accessible via the ``raw`` property.

    Returns
    -------
    RawModelData
        Assembled raw model data.
    """
    self._log.info("load_start")
    self._source.load_data()
    self._validate_source()

    self._raw = self._build_raw_model()
    self._log.info("load_done", facilities=len(self._raw.facilities))
    return self._raw


def _validate_source(self) -> None:
    """Validate source shape with Pandera schemas."""
    StationsSourceSchema.validate(self._source.df_stations)
    if self._config.build_observations:
        TripsSourceSchema.validate(self._source.df_trips)

    depots = _nonempty_df(self._source, "df_depots")
    if depots is not None:
        DepotsSourceSchema.validate(depots)

    resources = _nonempty_df(self._source, "df_resources")
    if resources is not None:
        ResourcesSourceSchema.validate(resources)

    self._log.debug("source_validated")

def _build_raw_model(self) -> RawModelData:
    """Assemble ``RawModelData`` from source DataFrames.

    Returns
    -------
    RawModelData
        Fully assembled (but not yet resolved) model data.
    """
    temporal = self._build_temporal()
    entities = self._build_entities()
    behavior = self._build_behavior(entities)
    distance_data = self._build_distance_matrix(entities) if self._config.build_edges else {}
    resources = self._build_resources(entities)

    observations: dict[str, pd.DataFrame | None] = {}
    if self._config.build_observations:
        observations = self._build_observations(entities)

    registry = AttributeRegistry()
    node_params = self._build_node_parameters(registry)
    self._register_facility_costs(registry, temporal)
    self._register_resource_costs(registry, entities)

    all_tables = {
        **temporal,
        **entities.tables,
        **behavior,
        **distance_data,
        **node_params,
        **resources,
        **{k: v for k, v in observations.items() if v is not None},
    }
    return RawModelData(
        **{k: v for k, v in all_tables.items() if v is not None},
        attributes=registry,
    )